# Stage 18: Matched-Direction Decomposition

Stage 17 revealed that while the lowest 10% denominator tail contributes to the moment-pathway gain, removing it only drops the gain by 5-15%. This implies that the specific directional geometry of the MoQ perturbation *within* the subspaces is driving the majority of the amplification.

This notebook decomposes the natural MoQ perturbation $u_{\text{nat}}$ and compares it to a randomized control. We measure:
1. $r(u_{\text{nat}})$: The full natural gain.
2. $r(u_N)$: The gain from the non-tail component only.
3. $r(u_T)$: The gain from the tail component only.
4. $r(u_{\text{matched}})$: The median gain of 50 random perturbations that have exactly the same $F_{10}$ tail-energy fraction as $u_{\text{nat}}$.

If $r(u_{\text{nat}}) \gg r(u_{\text{matched}})$, we will have definitively proven that MoQ's instability is caused by its precise directional alignment with the optimizer's scale geometry, not just coarse localization.

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

sys.path.append('.')
from official_soap import SOAP


In [ ]:
class InstrumentedOfficialSOAPStage18(SOAP):
    def __init__(self, params, lr=1e-3, betas=(0.95, 0.95), shampoo_beta=-1, 
                 eps=1e-8, weight_decay=0.0, precondition_frequency=10, 
                 max_precond_dim=10000, merge_dims=False, precondition_1d=False,
                 normalize_grads=False, data_format='channels_first', correct_bias=True,
                 mu=0.0, damping=1e-3):
        super().__init__(params, lr=lr, betas=betas, shampoo_beta=shampoo_beta, 
                         eps=eps, weight_decay=weight_decay, 
                         precondition_frequency=precondition_frequency,
                         max_precond_dim=max_precond_dim, merge_dims=merge_dims, 
                         precondition_1d=precondition_1d, normalize_grads=normalize_grads, 
                         data_format=data_format, correct_bias=correct_bias)
        self.mu = mu
        self.damping = damping
        self.layer_diagnostics = []
        self.target_steps = {50, 150, 300, 450}
        self.target_layers = {0, 4, 8}

    def compute_matched_direction_decomposition(self, raw_g, moq_g, state, group, p):
        if "exp_avg" not in state or "Q" not in state or state["Q"] is None:
            return None
            
        proj_raw = self.project(raw_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        proj_moq = self.project(moq_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        b1, b2 = group["betas"]
        m_old, v_old = state["exp_avg"], state["exp_avg_sq"]
        
        m_raw = m_old * b1 + proj_raw * (1.0 - b1)
        m_moq = m_old * b1 + proj_moq * (1.0 - b1)
        v_raw = v_old * b2 + proj_raw.square() * (1.0 - b2)
        
        denom_raw = v_raw.sqrt() + group["eps"]
        D_inv = 1.0 / denom_raw
        
        m_raw_norm = float(torch.linalg.vector_norm(m_raw))
        q_m = m_raw / m_raw_norm if m_raw_norm > 1e-12 else torch.zeros_like(m_raw)
        
        z_m = D_inv * q_m
        norm_z_m = float(torch.linalg.vector_norm(z_m))
        
        # Natural perturbation u_nat
        diff_m = m_moq - m_raw
        diff_m_norm = float(torch.linalg.vector_norm(diff_m))
        if diff_m_norm < 1e-12:
            return None
        u_nat = diff_m / diff_m_norm
        
        z_nat = D_inv * u_nat
        norm_z_nat = float(torch.linalg.vector_norm(z_nat))
        r_nat = norm_z_nat / (norm_z_m + 1e-12)
        
        denom_flat = denom_raw.flatten()
        k10 = max(1, int(0.10 * denom_flat.numel()))
        _, indices10 = torch.topk(denom_flat, k10, largest=False)
        
        tail_mask = torch.zeros_like(denom_flat, dtype=torch.bool)
        tail_mask[indices10] = True
        tail_mask_shaped = tail_mask.view(denom_raw.shape)
        
        z_nat_sq = z_nat.flatten()**2
        sum_z_nat_sq = float(torch.sum(z_nat_sq)) + 1e-12
        F10_nat = float(torch.sum(z_nat_sq[indices10])) / sum_z_nat_sq
        
        # Decompose u_nat
        u_nat_T = torch.where(tail_mask_shaped, u_nat, torch.zeros_like(u_nat))
        u_nat_N = torch.where(~tail_mask_shaped, u_nat, torch.zeros_like(u_nat))
        
        norm_u_nat_T_sq = float(torch.sum(u_nat_T**2))
        norm_u_nat_N_sq = float(torch.sum(u_nat_N**2))
        
        if norm_u_nat_T_sq < 1e-12 or norm_u_nat_N_sq < 1e-12:
            return None
            
        u_T = u_nat_T / math.sqrt(norm_u_nat_T_sq)
        u_N = u_nat_N / math.sqrt(norm_u_nat_N_sq)
        
        r_u_T = float(torch.linalg.vector_norm(D_inv * u_T)) / (norm_z_m + 1e-12)
        r_u_N = float(torch.linalg.vector_norm(D_inv * u_N)) / (norm_z_m + 1e-12)
        
        # Matched random controls (50 reps)
        f_target = F10_nat
        reps = 50
        r_matched_list = []
        
        for _ in range(reps):
            u_rand = torch.randn_like(denom_raw)
            u_rand_T = torch.where(tail_mask_shaped, u_rand, torch.zeros_like(u_rand))
            u_rand_N = torch.where(~tail_mask_shaped, u_rand, torch.zeros_like(u_rand))
            
            norm_rand_T_sq = float(torch.sum(u_rand_T**2))
            norm_rand_N_sq = float(torch.sum(u_rand_N**2))
            
            if norm_rand_T_sq < 1e-12 or norm_rand_N_sq < 1e-12:
                continue
                
            E_T = float(torch.sum((D_inv * u_rand_T)**2))
            E_N = float(torch.sum((D_inv * u_rand_N)**2))
            
            if f_target == 0.0:
                c_T = 0.0
                c_N = 1.0 / math.sqrt(norm_rand_N_sq + 1e-12)
            elif f_target == 1.0:
                c_T = 1.0 / math.sqrt(norm_rand_T_sq + 1e-12)
                c_N = 0.0
            else:
                ratio = (f_target * E_N) / ((1.0 - f_target) * E_T + 1e-12)
                c_N_sq = 1.0 / (ratio * norm_rand_T_sq + norm_rand_N_sq + 1e-12)
                c_T_sq = c_N_sq * ratio
                c_N = math.sqrt(c_N_sq)
                c_T = math.sqrt(c_T_sq)
                
            u_matched = c_T * u_rand_T + c_N * u_rand_N
            r_matched = float(torch.linalg.vector_norm(D_inv * u_matched)) / (norm_z_m + 1e-12)
            r_matched_list.append(r_matched)
            
        r_matched_median = float(np.median(r_matched_list)) if r_matched_list else 0.0
        
        return {
            "F10_nat": F10_nat,
            "r_nat": r_nat,
            "r_u_N": r_u_N,
            "r_u_T": r_u_T,
            "r_matched": r_matched_median
        }

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.layer_diagnostics = []
        
        current_step = 0
        if len(self.param_groups) > 0 and len(self.param_groups[0]['params']) > 0:
            p_sample = self.param_groups[0]['params'][0]
            if 'step' in self.state[p_sample]:
                current_step = self.state[p_sample]['step'] + 1
        else:
            current_step = 1
        
        moq_grads = []
        raw_grads = []
        
        idx = 0
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    idx += 1
                    continue
                    
                layer_idx = idx
                state = self.state[p]
                raw_g = p.grad.clone()
                raw_grads.append(raw_g)
                
                if 'moq_initialized' not in state:
                    state['moq_initialized'] = True
                    state['prev_raw_g'] = raw_g.clone()
                    g_moq = raw_g.clone()
                    state['moq_matched_direction'] = None
                else:
                    g_moq = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    state['prev_raw_g'].copy_(raw_g)
                    
                    if current_step in self.target_steps and layer_idx in self.target_layers:
                        if 'Q' in state and state['Q'] is not None:
                            data = self.compute_matched_direction_decomposition(raw_g, g_moq, state, group, p)
                            state['moq_matched_direction'] = data
                        else:
                            state['moq_matched_direction'] = None
                    else:
                        state['moq_matched_direction'] = None
                        
                p.grad.copy_(g_moq)
                moq_grads.append(g_moq)
                idx += 1
                
        super().step()

        idx = 0
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    idx += 1
                    continue
                
                layer_idx = idx
                state = self.state[p]
                
                if current_step in self.target_steps and layer_idx in self.target_layers and state.get('moq_matched_direction') is not None:
                    diag = {
                        "step": current_step,
                        "layer": layer_idx,
                        "is_weight": p.dim() > 1, 
                        "moq_matched_direction": state['moq_matched_direction']
                    }
                    self.layer_diagnostics.append(diag)
                    
                idx += 1
                
        return loss

    def _get_orthogonal_matrix(self, mat_list):
        final = []
        for m in mat_list:
            if len(m) == 0:
                final.append([])
                continue
            m64 = m.to(torch.float64)
            m64 = 0.5 * (m64 + m64.T)
            _, Q = torch.linalg.eigh(m64 + 1e-30*torch.eye(m.shape[0], device=m.device))
            Q = torch.flip(Q, [1])
            final.append(Q.to(m.dtype))
        return final

    def _project(self, grad, Q_list):
        for mat in Q_list:
            if len(mat) > 0:
                grad = torch.tensordot(grad, mat, dims=[[0], [0]])
            else:
                permute_order = list(range(1, len(grad.shape))) + [0]
                grad = grad.permute(permute_order)
        return grad


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None

    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule


In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_stage18_audit(mu: float, K: int=1, steps: int=500, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule_long.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
        if len(schedule) < steps:
            schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
            torch.save(schedule, schedule_path)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    opt = InstrumentedOfficialSOAPStage18(
        model.parameters(), 
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        mu=mu, 
        damping=1e-3
    )

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        if len(opt.layer_diagnostics) > 0:
            history.extend(opt.layer_diagnostics)
        
        if step % 50 == 0 or step == 1:
            err = relative_l2(model, device)
            print(f"Step {step:3d} | Loss {loss.item():.4e} | RelL2 {err:.4e}")

    return history


### Stage 18: Matched-Direction Decomposition Sweep

In [ ]:
for K in [20, 10]:
    print(f"\n{'='*40}\nRunning K={K} Matched-Direction Decomposition...\n{'='*40}")
    audit_controlled = run_stage18_audit(mu=0.5, K=K, steps=500)
    with open(f"stage18_K{K}_soap_moq.json", "w") as f:
        json.dump(audit_controlled, f, indent=2)
